In [19]:
import os
import logging
import pandas as pd


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("pipeline.log", mode="w"),
        logging.StreamHandler()                         
    ]
)

logging.info("Starting the traffic data pipeline.")


def load_csv_data(file_name):
    logging.info("Attempting to load the CSV file.")
    
    
    if not os.path.exists(file_name):
        logging.error("Error: The CSV file was not found in this folder.")
        return None

    
    try:
        data = pd.read_csv(file_name)
        logging.info("Success: CSV file loaded into memory.")
        return data
        
    except pd.errors.ParserError:
        logging.error("Error: The CSV file formatting is corrupted.")
        return None
    except Exception as e:
        logging.error(f"An unexpected error happened: {e}")
        return None


def clean_and_transform(df):
    if df is None:
        logging.warning("No data found to process. Stopping here.")
        return None

    logging.info("Starting data checks.")

   
    missing_data = df.isnull().sum()
    logging.info("Checking for missing values in each column:")
    for column, count in missing_data.items():
        if count > 0:
            logging.warning(f"Column '{column}' has {count} missing rows.")

    
    logging.info("Checking column data types.")

    
    logging.info("Extracting hour from date_time column.")
    try:
        df['date_time'] = pd.to_datetime(df['date_time'])
        df['hour'] = df['date_time'].dt.hour
    except Exception as e:
        logging.error(f"Could not convert date_time: {e}")

  
    logging.info("Converting temperature from Kelvin to Celsius.")
    df['temp_celsius'] = df['temp'] - 273.15

    
    logging.info("Creating traffic category column.")
    def check_volume(volume):
        if volume > 5500:
            return 'High'
        elif volume >= 2500:
            return 'Medium'
        else:
            return 'Low'

    df['traffic_category'] = df['traffic_volume'].apply(check_volume)
    
    logging.info("All data pipeline transformations completed successfully.")
    return df


if __name__ == "__main__":
   
    my_file = "Metro_Interstate_Traffic_Volume.csv"
    raw_df = load_csv_data(my_file)
    final_df = clean_and_transform(raw_df)


2026-09-24 16:29:58,069 - INFO - [1299295068.py:15] - Starting the traffic data pipeline.
2026-09-24 16:29:58,070 - INFO - [1299295068.py:19] - Attempting to load the CSV file.
2026-09-24 16:29:58,110 - INFO - [1299295068.py:29] - Success: CSV file loaded into memory.
2026-09-24 16:29:58,111 - INFO - [1299295068.py:45] - Starting data checks.
2026-09-24 16:29:58,116 - INFO - [1299295068.py:49] - Checking for missing values in each column:
2026-09-24 16:29:58,116 - WARNING - [1299295068.py:52] - Column 'holiday' has 48143 missing rows.
2026-09-24 16:29:58,117 - INFO - [1299295068.py:55] - Checking column data types.
2026-09-24 16:29:58,118 - INFO - [1299295068.py:58] - Extracting hour from date_time column.
2026-09-24 16:29:58,134 - INFO - [1299295068.py:66] - Converting temperature from Kelvin to Celsius.
2026-09-24 16:29:58,135 - INFO - [1299295068.py:70] - Creating traffic category column.
2026-09-24 16:29:58,141 - INFO - [1299295068.py:81] - All data pipeline transformations complet